In [1]:
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
df = pd.read_csv("../data/processed/reviews_with_sentiment.csv")
print(f"Loaded {len(df)} reviews.")
df.head()

Loaded 968 reviews.


,review,rating,date,bank,source,sentiment_label,sentiment_score
0,good,2,2026-07-28,Commercial Bank of Ethiopia,Google Play,positive,0.999816
1,ok. easly updare,5,2026-07-27,Commercial Bank of Ethiopia,Google Play,positive,0.997430
2,Almost Perfect! Easily one of the best and sim...,4,2026-07-27,Commercial Bank of Ethiopia,Google Play,positive,0.810805
3,best apps,1,2026-07-27,Commercial Bank of Ethiopia,Google Play,positive,0.999785
4,i hite yoy,5,2026-07-27,Commercial Bank of Ethiopia,Google Play,positive,0.967496


### Extract Keywords with TF-IDF

In [3]:
# Initialize TF-IDF Vectorizer to find important bigrams and trigrams
vectorizer = TfidfVectorizer(max_features=50, stop_words='english', ngram_range=(1, 3))
X = vectorizer.fit_transform(df['review'].astype(str))

# Get top keywords
keywords = vectorizer.get_feature_names_out()
print("Top Keywords/N-grams across reviews:")
print(keywords[:25])

Top Keywords/N-grams across reviews:
['account' 'amole' 'app' 'application' 'apps' 'bad' 'balance' 'bank'
 'banking' 'banking app' 'best' 'best app' 'better' 'boa' 'cbe' 'dashen'
 'dashen bank' 'doesn' 'doesn work' 'don' 'easy' 'ethiopia' 'fast' 'fix'
 'good']


## Theme Grouping Logic


We group extracted keywords into 4 core business themes based on semantic similarity and the actual vocabulary found in the scraped bank reviews:

Account Access Issues: Keywords like "account", "login", "otp", "password", "sign", "pin".

Transaction Performance: Keywords like "balance", "transfer", "slow", "pending", "fail", "doesn work".

UI & Design & App Stability: Keywords like "app", "application", "apps", "update", "fix", "better".

Customer Support & Bank Specifics: Keywords like "service", "support", "boa", "cbe", "dashen", "amole".









### Theme Mapping

In [ ]:
def assign_theme(text):
    text = str(text).lower()
    if any(k in text for k in ['login', 'otp', 'password', 'sign', 'account', 'pin', 'access']):
        return 'Account Access Issues'
    elif any(k in text for k in ['transfer', 'slow', 'pending', 'fail', 'transaction', 'balance', 'doesn work', 'working']):
        return 'Transaction Performance'
    elif any(k in text for k in ['app', 'application', 'apps', 'update', 'fix', 'ui', 'design', 'version', 'better']):
        return 'UI & Design & App Stability'
    elif any(k in text for k in ['service', 'support', 'call', 'help', 'feature', 'fingerprint', 'boa', 'cbe', 'dashen', 'amole']):
        return 'Customer Support & Bank Specifics'
    else:
        return 'General Feedback'

df['identified_theme'] = df['review'].apply(assign_theme)

print(df.groupby(['bank', 'identified_theme']).size())

bank                         identified_theme                 
Bank of Abyssinia            Account Access Issues                 24
                             Customer Support & Bank Specifics     16
                             General Feedback                     147
                             Transaction Performance               36
                             UI & Design & App Stability          105
Commercial Bank of Ethiopia  Account Access Issues                 30
                             Customer Support & Bank Specifics     13
                             General Feedback                     135
                             Transaction Performance               20
                             UI & Design & App Stability          113
Dashen Bank                  Account Access Issues                 32
                             Customer Support & Bank Specifics     18
                             General Feedback                     144
                           

### Save Final Processed Dataset

In [6]:
import os
output_path = "../data/processed/reviews_final_processed.csv"
df.to_csv(output_path, index=False)
print(f"Saved final processed dataset with themes to {output_path}")

Saved final processed dataset with themes to ../data/processed/reviews_final_processed.csv
